# QUN Kaggle Render Worker — Image MVP

Личный GPU-узел QUN для **генерации артов** через QUN Studio.

## Как это работает
1. QUN Studio создаёт задание в приватной очереди QUN Compute.
2. Этот ноутбук забирает задание с помощью уже существующего `comfyui_worker.py`.
3. ComfyUI генерирует PNG на GPU Kaggle.
4. Worker загружает результат в приватный QUN Storage и регистрирует материал в QUN Library.

**Не нужен публичный туннель.** ComfyUI слушает только `127.0.0.1`; ноутбук сам обращается к QUN через HTTPS.

## Перед запуском
- В Kaggle включи `Settings → Accelerator → GPU` (если доступен выбор — T4x2) и `Internet → On`.
- В Hugging Face открой карточку Stable Diffusion 1.5, прими условия модели и добавь Read token в Kaggle Secrets с именем `HF_TOKEN`.
- В отдельной вкладке войди в QUN Studio под своим QUN ID.

## Ограничения первой версии
- Сейчас подключён **только text-to-image** через Stable Diffusion 1.5, 512×512. Видео-модель подключим отдельно после успешного теста артов.
- Kaggle-сессия временная. Пока последняя ячейка Worker выполняется, узел онлайн; после остановки/истечения сессии задания останутся в очереди.
- Kaggle Terms ограничивают использование личными некоммерческими задачами. Не используй этот бесплатный ноутбук для коммерческого сервиса, клиентских задач или вычислений для других пользователей.
- Не запускай `Save & Run All` с активным Worker. Не публикуй ноутбук с выводом, содержащим `QUN_NODE_TOKEN` или `HF_TOKEN`.

In [ ]:
# 1. Проверка среды Kaggle и GPU
from pathlib import Path
import os, re, sys, json, time, shutil, subprocess, urllib.request

try:
    import torch
except ImportError as exc:
    raise RuntimeError("В среде Kaggle не найден PyTorch. Создай новый Notebook с актуальным образом Kaggle.") from exc

if not torch.cuda.is_available():
    raise RuntimeError("GPU не подключён. Открой Settings → Accelerator → GPU и перезапусти сессию.")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {props.name} · {props.total_memory / (1024**3):.1f} GiB")

# QUN worker uses one GPU for the first image-generation proof.
print("Working GPU:", torch.cuda.get_device_name(0))
print("Internet setting: must be enabled in Kaggle Notebook settings.")
WORK = Path("/kaggle/working/qun-kaggle-worker")
WORK.mkdir(parents=True, exist_ok=True)


In [ ]:
# 2. Получение QUN Worker и установка ComfyUI
QUN_REPO = Path("/kaggle/working/QUN")
COMFYUI_DIR = Path("/kaggle/working/ComfyUI")

if not QUN_REPO.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/Serega40in/QUN.git", str(QUN_REPO)],
        check=True
    )
else:
    print("QUN repository already exists:", QUN_REPO)

if not COMFYUI_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFYUI_DIR)],
        check=True
    )
else:
    print("ComfyUI already exists:", COMFYUI_DIR)

# Keep Kaggle's preinstalled CUDA-enabled PyTorch. Install the remaining ComfyUI requirements.
requirements = COMFYUI_DIR / "requirements.txt"
filtered = []
skip_packages = {"torch", "torchvision", "torchaudio"}
for line in requirements.read_text(encoding="utf-8").splitlines():
    stripped = line.strip()
    if not stripped or stripped.startswith("#"):
        continue
    match = re.match(r"([A-Za-z0-9_.-]+)", stripped)
    package = match.group(1).lower().replace("_", "-") if match else ""
    if package in skip_packages:
        continue
    filtered.append(line)

filtered_file = WORK / "comfyui-requirements-kaggle.txt"
filtered_file.write_text("\n".join(filtered) + "\n", encoding="utf-8")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "-r", str(filtered_file)],
    check=True
)

print("ComfyUI dependencies installed. Existing PyTorch was kept.")


In [ ]:
# 3. Загрузка модели Stable Diffusion 1.5
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download

def kaggle_secret(name):
    try:
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

HF_TOKEN = kaggle_secret("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError(
        "Не найден Kaggle Secret HF_TOKEN. Прими условия модели на Hugging Face, "
        "создай Read token и добавь его в Kaggle → Add-ons → Secrets."
    )

MODEL_REPO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
MODEL_FILE = "v1-5-pruned-emaonly.safetensors"
MODEL_DIR = COMFYUI_DIR / "models" / "checkpoints"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = Path(hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    token=HF_TOKEN,
    local_dir=str(MODEL_DIR),
))
if not model_path.is_file() or model_path.stat().st_size < 1_000_000_000:
    raise RuntimeError("Файл checkpoint не загрузился полностью. Проверь доступ к модели и интернет.")

MODEL_NAME = model_path.name
print("Checkpoint ready:", MODEL_NAME)
print("Checkpoint size: %.2f GiB" % (model_path.stat().st_size / (1024**3)))
# Не выводим HF_TOKEN и не сохраняем его в файлы проекта.


In [ ]:
# 4. Подготовка проверенного ComfyUI API workflow для артов
# QUN Worker подставляет {{PROMPT}} и {{OUTPUT_PREFIX}} в эту фиксированную схему.
# Пользовательский запрос не может передавать произвольный workflow или путь к файлу.

WORKFLOW_PATH = WORK / "workflow-image-api.json"
workflow = {
    "4": {
        "class_type": "CheckpointLoaderSimple",
        "inputs": {"ckpt_name": MODEL_NAME}
    },
    "5": {
        "class_type": "EmptyLatentImage",
        "inputs": {"width": 512, "height": 512, "batch_size": 1}
    },
    "6": {
        "class_type": "CLIPTextEncode",
        "inputs": {
            "text": "{{PROMPT}}",
            "clip": ["4", 1]
        }
    },
    "7": {
        "class_type": "CLIPTextEncode",
        "inputs": {
            "text": "low quality, blurry, low detail, text, watermark, logo, malformed anatomy, extra fingers, duplicate subjects",
            "clip": ["4", 1]
        }
    },
    "3": {
        "class_type": "KSampler",
        "inputs": {
            "seed": 4061109,
            "steps": 28,
            "cfg": 7.0,
            "sampler_name": "euler",
            "scheduler": "normal",
            "denoise": 1.0,
            "model": ["4", 0],
            "positive": ["6", 0],
            "negative": ["7", 0],
            "latent_image": ["5", 0]
        }
    },
    "8": {
        "class_type": "VAEDecode",
        "inputs": {"samples": ["3", 0], "vae": ["4", 2]}
    },
    "9": {
        "class_type": "SaveImage",
        "inputs": {"filename_prefix": "{{OUTPUT_PREFIX}}", "images": ["8", 0]}
    }
}
WORKFLOW_PATH.write_text(json.dumps(workflow, ensure_ascii=False, indent=2), encoding="utf-8")
print("Image workflow saved:", WORKFLOW_PATH)


In [ ]:
# 5. Подключение этого GPU к твоему QUN ID
# Один раз создай код в QUN Studio → Подключить GPU-узел.
# Код одноразовый и действует 15 минут.

SUPABASE_URL = "https://qdxjltqlnphvzafapncm.supabase.co"
SUPABASE_PUBLISHABLE_KEY = "sb_publishable_aCUwbGYj_kUU8nVUu1kWmQ_-d-8fp8L"
NODE_FILE = Path("/kaggle/working/.config/qun/render-node.json")
NODE_FILE.parent.mkdir(parents=True, exist_ok=True)

os.environ["QUN_SUPABASE_URL"] = SUPABASE_URL
os.environ["QUN_SUPABASE_PUBLISHABLE_KEY"] = SUPABASE_PUBLISHABLE_KEY
os.environ["QUN_NODE_PLATFORM"] = "linux-gpu"
os.environ["QUN_NODE_REGISTRY_FILE"] = str(NODE_FILE)

saved_node_id = kaggle_secret("QUN_NODE_ID")
saved_node_token = kaggle_secret("QUN_NODE_TOKEN")

if saved_node_id and saved_node_token:
    # Reuse the same owner-bound node on later Kaggle sessions.
    credentials = {
        "nodeId": saved_node_id,
        "nodeToken": saved_node_token,
        "platform": "linux-gpu"
    }
    NODE_FILE.write_text(json.dumps(credentials, indent=2) + "\n", encoding="utf-8")
    try:
        NODE_FILE.chmod(0o600)
    except OSError:
        pass
    print("Используем сохранённый QUN Render Node:", saved_node_id)
else:
    print("Открой QUN Studio → «Подключить GPU-узел» → «Создать код подключения».")
    print("Введи одноразовый код ниже. Он будет использован только для регистрации этого узла.")
    from getpass import getpass
    enrollment_code = getpass("Одноразовый код QUN: ").strip()
    if len(enrollment_code) < 20:
        raise RuntimeError("Код пустой или слишком короткий. Создай новый код в QUN Studio.")

    sys.path.insert(0, str(QUN_REPO / "docs" / "studio"))
    import register_gpu_worker as node_registry
    registration = node_registry.request_registration("QUN Kaggle GPU", enrollment_code)
    node_registry.save_credentials(registration, NODE_FILE)

    print("Узел зарегистрирован:", registration["nodeId"])
    print("\nДля повторного использования добавь эти два значения в Kaggle → Add-ons → Secrets:")
    print("QUN_NODE_ID =", registration["nodeId"])
    print("QUN_NODE_TOKEN =", registration["nodeToken"])
    print("После копирования обязательно очисти вывод этой ячейки. Не публикуй и не сохраняй версию ноутбука с токеном в выводе.")

node_data = json.loads(NODE_FILE.read_text(encoding="utf-8"))
if not node_data.get("nodeId") or not node_data.get("nodeToken"):
    raise RuntimeError("Файл регистрации QUN Node неполный.")
print("QUN Render Node is ready. Platform: linux-gpu")


In [ ]:
# 6. Запуск ComfyUI локально (только 127.0.0.1)
COMFY_LOG_PATH = Path("/kaggle/working/comfyui.log")
COMFY_LOG = open(COMFY_LOG_PATH, "w", encoding="utf-8")

comfy_process = subprocess.Popen(
    [
        sys.executable, "main.py",
        "--listen", "127.0.0.1",
        "--port", "8188"
    ],
    cwd=str(COMFYUI_DIR),
    stdout=COMFY_LOG,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

COMFY_URL = "http://127.0.0.1:8188"
for attempt in range(60):
    if comfy_process.poll() is not None:
        raise RuntimeError("ComfyUI завершился при запуске. Последние строки смотри в /kaggle/working/comfyui.log.")
    try:
        with urllib.request.urlopen(COMFY_URL + "/system_stats", timeout=5) as response:
            stats = json.loads(response.read().decode("utf-8"))
        print("ComfyUI API is ready:", COMFY_URL)
        break
    except Exception:
        time.sleep(3)
else:
    raise RuntimeError("ComfyUI не ответил за 3 минуты. Проверь /kaggle/working/comfyui.log.")

print("ComfyUI PID:", comfy_process.pid)
print("ComfyUI открыт только внутри Kaggle-сессии; публичный порт и туннель не создаются.")


In [ ]:
# 7. Запуск QUN Kaggle Worker
# Пока эта ячейка выполняется, Worker отправляет heartbeat и забирает задания из QUN Compute.
# Открой QUN Studio в другой вкладке и создай тестовое задание типа «Генерация изображения».

STUDIO_DIR = QUN_REPO / "docs" / "studio"
WORKER_SCRIPT = STUDIO_DIR / "comfyui_worker.py"

worker_env = os.environ.copy()
worker_env.update({
    "QUN_SUPABASE_URL": SUPABASE_URL,
    "QUN_SUPABASE_PUBLISHABLE_KEY": SUPABASE_PUBLISHABLE_KEY,
    "QUN_NODE_REGISTRY_FILE": str(NODE_FILE),
    "QUN_NODE_PLATFORM": "linux-gpu",
    "QUN_COMFYUI_URL": COMFY_URL,
    "QUN_COMFYUI_WORKFLOW_IMAGE": str(WORKFLOW_PATH),
    "QUN_COMFYUI_OUTPUT_DIR": str(COMFYUI_DIR / "output"),
    "QUN_POLL_SECONDS": "8",
    "QUN_JOB_TIMEOUT_SECONDS": "900"
})

if not comfy_process or comfy_process.poll() is not None:
    raise RuntimeError("ComfyUI не запущен. Сначала выполни ячейку 6.")

# Apply the runtime configuration before importing the worker module.
os.environ.update(worker_env)
sys.path.insert(0, str(STUDIO_DIR))
import importlib
if "comfyui_worker" in sys.modules:
    comfyui_worker = importlib.reload(sys.modules["comfyui_worker"])
else:
    import comfyui_worker

print("QUN Kaggle Worker запущен.")
print("1) В QUN Studio сначала выбери «Проверка очереди».")
print("2) После успешной проверки создай задание «Генерация изображения».")
print("3) Готовый PNG появится в QUN Studio → Библиотека.")
print("Оставь эту ячейку выполняться. Для остановки нажми Interrupt.")
try:
    comfyui_worker.main()
except KeyboardInterrupt:
    print("QUN Worker остановлен пользователем.")


In [ ]:
# 8. Остановка ComfyUI (выполни после остановки Worker)
try:
    if "comfy_process" in globals() and comfy_process and comfy_process.poll() is None:
        comfy_process.terminate()
        try:
            comfy_process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            comfy_process.kill()
        print("ComfyUI остановлен.")
    else:
        print("ComfyUI уже остановлен.")
finally:
    try:
        COMFY_LOG.close()
    except Exception:
        pass


## Проверка первого результата

1. Запусти ячейку 7 и дождись heartbeat узла.
2. В QUN Studio сначала отправь `health_check` («Проверка очереди»).
3. После успешной проверки отправь короткий prompt в режиме **Генерация изображения**.
4. Убедись, что задание перешло в `succeeded`, а PNG появился в QUN Library и открывается с телефона.
5. Останови Worker через Interrupt, затем останови ComfyUI ячейкой 8.

**Важно:** первый тест подтверждает только путь text-to-image. Не отправляй `video_generation` или `audio_generation` на этот MVP: видео- и аудио-workflow ещё не установлены.